#Building a RAG system that uses the latest LangChain package

Instead of using RetrievalQA, I implemented a custom LCEL-based RAG pipeline for greater flexibility and transparency.

## Installations

In [1]:
!pip install -q \
langchain==0.2.14 \
langchain-community==0.2.12 \
langchain-openai==0.1.22 \
faiss-cpu \
pymupdf \
tiktoken \
requests==2.32.4


In [2]:
import langchain

langchain.verbose = False

##OpenAI API Key

In [3]:
import os

os.environ["OPENAI_API_KEY"] = ""

##Loading Documents (PDF + TXT + Raw Text)

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os

project_path = "/content/drive/MyDrive/RAG_Project"

os.chdir(project_path)

print("Current directory:", os.getcwd())
print("Files in directory:")
print(os.listdir())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current directory: /content/drive/MyDrive/RAG_Project
Files in directory:
['sample_DS.pdf', 'sample_ML.pdf', 'sample_statistical_ml.pdf', 'sample_dl.pdf', 'sample_DS_07Nov2024.pdf', 'sample_txt_1.txt', 'sample_txt_2.txt']


In [5]:
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader
from langchain.schema import Document

documents = []

# Load PDFs
pdf_files = [
    "/content/drive/MyDrive/RAG_Project/sample_dl.pdf",
    "/content/drive/MyDrive/RAG_Project/sample_DS.pdf"
]
for file in pdf_files:
    loader = PyMuPDFLoader(file)
    documents.extend(loader.load())

# Load text files
txt_files = [
    "/content/drive/MyDrive/RAG_Project/sample_txt_1.txt",
    "/content/drive/MyDrive/RAG_Project/sample_txt_2.txt"
]
for file in txt_files:
    loader = TextLoader(file)
    documents.extend(loader.load())

# Raw text input
raw_texts = [
    "In December 2015, OpenAI was founded as a not for profit organization by Sam Altman, Elon Musk, Ilya Sutskever, Greg Brockman, Trevor Blackwell, Vicki Cheung, Andrej Karpathy, Durk Kingma, John Schulman, Pamela Vagata, and Wojciech Zaremba, with Sam Altman and Elon Musk as the co-chairs. A total of $1 billion in capital was pledged by Sam Altman, Greg Brockman, Elon Musk, Reid Hoffman, Jessica Livingston, Peter Thiel, Amazon Web Services (AWS), and Infosys. However, the actual capital collected significantly lagged pledges. According to company disclosures, only $130 million had been received by 2019.",
    "In 2019, OpenAI transitioned from non-profit to 'capped' for-profit, with the profit being capped at 100 times any investment. According to OpenAI, the capped-profit model allows OpenAI Global, LLC to legally attract investment from venture funds and, in addition, to grant employees stakes in the company. Many top researchers work for Google Brain, DeepMind, or Facebook, which offer equity that a nonprofit would be unable to match. Before the transition, OpenAI was legally required to publicly disclose the compensation of its top employees."
]

for text in raw_texts:
    documents.append(Document(page_content=text))

print(f"Total documents loaded: {len(documents)}")

Total documents loaded: 7


##Chunking Documents

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

splits = text_splitter.split_documents(documents)

print(f"Total chunks: {len(splits)}")

Total chunks: 36


##Creating Embeddings + Vector Store

In [7]:
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.from_documents(splits, embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

##Adding Memory

In [8]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

##Defining Prompt Template

In [9]:
from langchain.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Use the following context to answer the question in the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Use three sentences maximum.
Keep the answer as concise as possible. Always say "thanks for asking!" at the end of the answer.

Chat History:
{chat_history}

Context:
{context}

Question:
{question}
Helpful Answer:""")

##Building RAG Chain (LCEL instead of RetrievalQA)

In [12]:
from langchain_openai import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough

from langchain.globals import set_debug

set_debug(True)

llm = ChatOpenAI(model="gpt-5-mini", temperature=1)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
        "chat_history": lambda x: memory.load_memory_variables({})["chat_history"]
    }
    | prompt
    | llm
)

##Runnig Query - Storing Memory

In [13]:
def ask_question(question):
    response = rag_chain.invoke(question)

    # Save to memory
    memory.save_context(
        {"input": question},
        {"output": response.content}
    )

    return response.content

# Test
print(ask_question("What is discussed in the documents?"))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "What is discussed in the documents?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question,chat_history>] Entering Chain run with input:
{
  "input": "What is discussed in the documents?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question,chat_history> > chain:RunnableSequence] Entering Chain run with input:
{
  "input": "What is discussed in the documents?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question,chat_history> > chain:RunnablePassthrough] Entering Chain run with input:
{
  "input": "What is discussed in the documents?"
}
[chain/end] [chain:RunnableSequence > chain:RunnableParallel<context,question,chat_history> > chain:RunnablePassthrough] s] Exiting Chain run with output:
{
  "output": "What is discussed in the documents?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,questio

##Multi-turn Conversation Test

In [14]:
print(ask_question("Can you summarize that?"))
print(ask_question("Give more details about the first topic."))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Can you summarize that?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question,chat_history>] Entering Chain run with input:
{
  "input": "Can you summarize that?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question,chat_history> > chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Can you summarize that?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question,chat_history> > chain:RunnablePassthrough] Entering Chain run with input:
{
  "input": "Can you summarize that?"
}
[chain/end] [chain:RunnableSequence > chain:RunnableParallel<context,question,chat_history> > chain:RunnablePassthrough] s] Exiting Chain run with output:
{
  "output": "Can you summarize that?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question,chat_history> > chain:RunnableLambda] Entering Chain run w

##Inspecting Retrieved Documents (Debugging)

In [15]:
query = "What is discussed in the documents?"
docs = retriever.get_relevant_documents(query)

for i, doc in enumerate(docs):
    print(f"\n--- Document {i+1} ---\n")
    print(doc.page_content[:500])

/tmp/ipykernel_6735/3773761300.py:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  docs = retriever.get_relevant_documents(query)



--- Document 1 ---

Peter Thiel, Amazon Web Services (AWS), and Infosys. However, the actual capital collected significantly lagged pledges. According to company disclosures, only $130 million had been received by 2019.

--- Document 2 ---

Creative industries 'incredibly worried' about OpenAI-Disney deal

Warner settles lawsuit with AI music firm and launches joint venture

--- Document 3 ---

- **Text Classification**: BERT is used for classifying text into predefined categories, such as
spam detection in emails or sentiment analysis in product reviews.
- **Question Answering**: By understanding the context of a passage, BERT can accurately
answer questions posed by users, making it invaluable in virtual assistants and customer
support.
- **Named Entity Recognition (NER)**: BERT identifies and classifies entities in text, such as
